# Credit Card Transaction Data Fingerprinting Analysis

This notebook implements advanced data fingerprinting techniques on credit card transaction data to identify patterns, similarities, and potential re-use of transaction data.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 8)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

In [ ]:
# Load the dataset
df = pd.read_csv('credit_card_transactions.csv')

# Display basic information about the dataset
print("Dataset shape:", df.shape)
print("\nDataset info:")
df.info()

# Display first few rows
df.head(5)

In [ ]:
# Data cleaning and preprocessing
df_clean = df.copy()

# Convert date to datetime
df_clean['trans_date_trans_time'] = pd.to_datetime(df_clean['trans_date_trans_time'])

# Extract time features
df_clean['hour'] = df_clean['trans_date_trans_time'].dt.hour
df_clean['day_of_week'] = df_clean['trans_date_trans_time'].dt.dayofweek
df_clean['month'] = df_clean['trans_date_trans_time'].dt.month

# Handle categorical features
categorical_features = ['gender', 'category', 'job']
le = LabelEncoder()
for col in categorical_features:
    if col in df_clean.columns:
        df_clean[col] = le.fit_transform(df_clean[col].astype(str))

# Create a subset for fingerprinting analysis
fingerprint_features = ['amt', 'hour', 'day_of_week', 'category', 'gender', 'city_pop', 'job']

# Create the fingerprinting dataset
df_fingerprint = df_clean[fingerprint_features].copy()
print("Fingerprint dataset shape:", df_fingerprint.shape)
df_fingerprint.head()

In [ ]:
# Data analysis
print("Dataset statistics:")
df_fingerprint.describe()

In [ ]:
# Correlation matrix
plt.figure(figsize=(12, 10))
correlation_matrix = df_fingerprint.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Visualize transaction amount distribution
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

axes[0].hist(df_fingerprint['amt'], bins=50, alpha=0.7, color='blue')
axes[0].set_xlabel('Transaction Amount')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Transaction Amounts')

axes[1].boxplot(df_fingerprint['amt'])
axes[1].set_ylabel('Transaction Amount')
axes[1].set_title('Box Plot of Transaction Amounts')

plt.tight_layout()
plt.show()

In [ ]:
# Data fingerprinting using multiple techniques

# 1. Standardize the features
scaler = StandardScaler()
scaled_data = scaler.fit_transform(df_fingerprint)

# 2. Apply PCA
pca = PCA()
pca_result = pca.fit_transform(scaled_data)

# Explained variance
plt.figure(figsize=(12, 6))
plt.plot(range(1, len(pca.explained_variance_ratio_) + 1), np.cumsum(pca.explained_variance_ratio_))
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('Explained Variance by Principal Components')
plt.grid(True)
plt.show()

print("Explained variance ratio:")
cumulative_variance = 0
for i, ratio in enumerate(pca.explained_variance_ratio_):
    cumulative_variance += ratio
    print(f"Component {i+1}: {ratio:.4f} ({cumulative_variance:.4f} total)")

In [ ]:
# Clustering using KMeans
# Determine optimal number of clusters using elbow method
wcss = []
K_range = range(1, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(scaled_data)
    wcss.append(kmeans.inertia_)

plt.figure(figsize=(10, 6))
plt.plot(K_range, wcss, 'bo-')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Within-Cluster Sum of Squares (WCSS)')
plt.title('Elbow Method for Optimal k')
plt.grid(True)
plt.show()

# Run KMeans with k=4 as a starting point
kmeans = KMeans(n_clusters=4, random_state=42)
clusters = kmeans.fit_predict(scaled_data)

# Add cluster labels to dataframe
df_clean['cluster'] = clusters

# Analyze clusters
cluster_analysis = df_clean.groupby('cluster').agg({
    'amt': ['mean', 'std', 'count'],
    'hour': 'mean',
    'category': lambda x: x.mode().iloc[0] if not x.mode().empty else 'N/A'
}).round(2)

cluster_analysis

In [ ]:
# Visualize clusters in 2D space using PCA
pca_2d = PCA(n_components=2)
pca_2d_result = pca_2d.fit_transform(scaled_data)

plt.figure(figsize=(12, 10))
scatter = plt.scatter(pca_2d_result[:, 0], pca_2d_result[:, 1], c=clusters, cmap='viridis', alpha=0.6, s=50)
plt.xlabel(f'First Principal Component (Explains {pca_2d.explained_variance_ratio_[0]:.2%} variance)')
plt.ylabel(f'Second Principal Component (Explains {pca_2d.explained_variance_ratio_[1]:.2%} variance)')
plt.title('Transaction Clusters (2D PCA Projection)')
plt.colorbar(scatter)
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Identify transaction patterns that could indicate data fingerprinting

# Create fingerprint using multiple features
def create_fingerprint(row):
    return f"{row['amt']:.0f}_{row['hour']}_{row['category']}_{row['gender']}"

df_clean['fingerprint'] = df_clean.apply(create_fingerprint, axis=1)

# Find transactions with similar fingerprints (potential data reuse)
fingerprint_counts = df_clean['fingerprint'].value_counts()
similar_fingerprints = fingerprint_counts[fingerprint_counts > 1]

print(f"Found {len(similar_fingerprints)} patterns with multiple occurrences:")
for fingerprint, count in similar_fingerprints.head(10).items():
    print(f"  {fingerprint}: {count} occurrences")

In [ ]:
# Show examples of similar transactions
# Get first few transactions with repeated fingerprint patterns
sample_transactions = []
for fingerprint in similar_fingerprints.head(3).index:
    matching_rows = df_clean[df_clean['fingerprint'] == fingerprint]
    sample_transactions.append(matching_rows.head(2))  # Show first 2 matching transactions

# Display a couple of examples
for i, transactions in enumerate(sample_transactions):
    print(f"\nExample {i+1} of similar transactions:")
    print(transactions[['fingerprint', 'amt', 'hour', 'category', 'gender', 'is_fraud', 'cc_num']])

In [ ]:
# Advanced fingerprinting: N-gram based fingerprints
print("\n=== Advanced Fingerprint Analysis ===")

# Use all relevant features as a string for text-based similarity
df_clean['full_fingerprint'] = df_clean[['amt', 'hour', 'category', 'gender']].apply(lambda x: '_'.join(x.astype(str)), axis=1)

# Analyze frequency of patterns
full_fingerprint_counts = df_clean['full_fingerprint'].value_counts()
print("Most common transaction patterns:")
print(full_fingerprint_counts.head(10))

# Calculate pattern similarity
print(f"\nTotal unique patterns: {len(full_fingerprint_counts)}")
print(f"Total transactions: {len(df_clean)}")
print(f"Pattern reuse rate: {1 - len(full_fingerprint_counts)/len(df_clean):.2%}")

In [ ]:
# Create a hierarchical clustering dendrogram for better visualization
print("\n=== Hierarchical Clustering Dendrogram ===")

# Take a sample of 500 transactions for better visualization
sample_size = min(500, len(scaled_data))
sample_scaled = scaled_data[:sample_size]

# Create linkage matrix
from scipy.cluster.hierarchy import dendrogram, linkage
linkage_matrix = linkage(sample_scaled, method='ward')

# Plot dendrogram
plt.figure(figsize=(15, 8))
dendrogram(linkage_matrix, truncate_mode='lastp', p=30)
plt.title('Hierarchical Clustering Dendrogram (Sample of 500 transactions)')
plt.xlabel('Transaction Index')
plt.ylabel('Distance')
plt.show()

In [ ]:
# Summary of findings
print("=== DATA FINGERPRINTING ANALYSIS SUMMARY ===")
print(f"Total transactions: {len(df_clean)}")
print(f"Fraudulent transactions: {df_clean['is_fraud'].sum()}")
print(f"Feature set used for fingerprinting: {', '.join(fingerprint_features)}")
print(f"\nKey Insights:")
print("1. Transaction amount distributions show potential fraud patterns")
print("2. Clustering reveals distinct groups of transaction patterns")
print("3. Pattern reuse detection identifies potential data fingerprinting")
print("4. Time-based patterns may indicate data fingerprinting")
print("5. Pattern reuse suggests possible data duplication or fingerprinting")
print(f"6. {len(similar_fingerprints)} patterns occur multiple times")

print("\nData fingerprinting techniques used:")
print("- Dimensionality reduction with PCA")
print("- KMeans clustering with elbow method")
print("- Pattern-based fingerprint matching")
print("- Hierarchical clustering visualization")
print("- Multi-dimensional pattern analysis")
print("- Memory-efficient approach without large similarity matrices")